# Module 05 Theory: Validation Architecture & Baselines

Validation is the heartbeat of a Kaggle model. This notebook covers the framework for robust Stratified K-Fold cross-validation, baseline modeling, and the critical hunt for data leakage.

## 1. Robust Stratified K-Fold CV
Properly splitting data to ensure reliable performance estimates.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Mock Data
X = np.random.rand(100, 5)
y = np.random.randint(0, 2, 100)

oof_preds = np.zeros(len(y))
skf = StratifiedKFold(n_splits=5)

for train_idx, val_idx in skf.split(X, y):
    model = RandomForestClassifier().fit(X[train_idx], y[train_idx])
    oof_preds[val_idx] = model.predict_proba(X[val_idx])[:, 1]

print("OOF Predictions Shape:", oof_preds.shape)

## 2. Preventing Data Leakage
Crucial: preprocessing (Scaling/Imputing) must happen *within* the fold, not globally.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# INCORRECT (Leaky): Scaler fitted on whole dataset
# X_leaked = StandardScaler().fit_transform(X)

# CORRECT (Robust): Scaler fitted inside the loop on training fold only
for train_idx, val_idx in skf.split(X, y):
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    
    X_tr = imputer.fit_transform(X[train_idx])
    X_tr = scaler.fit_transform(X_tr)
    
    # Now transform validation using train stats
    X_val = imputer.transform(X[val_idx])
    X_val = scaler.transform(X_val)
    
    print("Fold processed: Imputed and Scaled using Train-Fold statistics.")
    break # Just for demo